# 05 — Reinforcement Learning: PPO Training
**Goal**: Train the SFT model using Proximal Policy Optimization (PPO) with execution-guided sandbox rewards and KL divergence penalties (KL $\beta=0.02$, LR $1\times 10^{-6}$). Produces `./checkpoints/ppo/final`.

---

## Step 1: Environment & Universal Path Resolution

In [ ]:
!pip install -q --no-deps trl==0.9.6

import sys, os, shutil

# Universal Path Resolution & Auto-Copy for Kaggle
def prepare_kaggle_src():
    curr = os.path.abspath(os.getcwd())
    if os.path.exists(os.path.join(curr, 'src', 'models', 'loader.py')):
        print(f"Using local 'src' directory at {curr}")
        return curr
    
    if os.path.exists('/kaggle/input'):
        working_src = '/kaggle/working/src'
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'models' in dirs and os.path.exists(os.path.join(root, 'models', 'loader.py')):
                if os.path.exists(working_src):
                    shutil.rmtree(working_src)
                shutil.copytree(root, working_src)
                print(f"Copied 'src' from {root} to {working_src}")
                break
            elif 'src' in dirs and os.path.exists(os.path.join(root, 'src', 'models', 'loader.py')):
                src_dir = os.path.join(root, 'src')
                if os.path.exists(working_src):
                    shutil.rmtree(working_src)
                shutil.copytree(src_dir, working_src)
                print(f"Copied 'src' from {src_dir} to {working_src}")
                break
    
    return '/kaggle/working' if os.path.exists('/kaggle/working/src') else curr

repo_root = prepare_kaggle_src()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print(f"Project path added: {repo_root}")

import time
import torch
from datasets import load_dataset
from transformers import AutoTokenizer
from src.training.ppo import run_ppo_training
from src.utils.checkpoint import auto_checkpoint, guard_session_limit

os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Environment initialized!")

## Step 2: Load APPS Dataset & Tokenizer

In [ ]:
!pip uninstall -y torchao

MODEL_NAME = "deepseek-ai/deepseek-coder-1.3b-instruct"
SFT_CHECKPOINT = "./checkpoints/sft/final"

# Resolve SFT Checkpoint path on Kaggle input / working
if not os.path.exists(SFT_CHECKPOINT) and os.path.exists('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'adapter_config.json' in files or 'model.safetensors' in files:
            SFT_CHECKPOINT = root
            break

effective_model = SFT_CHECKPOINT if os.path.exists(SFT_CHECKPOINT) else MODEL_NAME
print(f"Using SFT model checkpoint: {effective_model}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading APPS training dataset for PPO via Parquet branch...")
apps = load_dataset('codeparrot/apps', revision='refs/convert/parquet', split='train[:1000]')
apps_clean = apps.filter(lambda x: len(x['solutions']) > 0).flatten_indices()

def tokenize_ppo(example):
    problem = example.get("question", example.get("prompt", ""))
    prompt_text = f"### Problem:\n{problem}\n\n### Solution:\n```python\n"
    tokens = tokenizer(prompt_text, truncation=True, max_length=512, padding="max_length")
    return {"input_ids": tokens["input_ids"], "query": prompt_text}

ppo_dataset = apps_clean.map(tokenize_ppo, remove_columns=apps_clean.column_names).flatten_indices()
print(f"Prepared {len(ppo_dataset)} tokenized APPS problems for PPO rollout.")

## Step 3: Run PPO Training with Auto-Checkpointing

In [ ]:
import gc, time, sys, torch

try:
    from trl import AutoModelForCausalLMWithValueHead
except ImportError:
    try:
        from trl.models import AutoModelForCausalLMWithValueHead
    except ImportError:
        from trl.models.modeling_value_head import AutoModelForCausalLMWithValueHead

try:
    from trl import PPOConfig, PPOTrainer
except ImportError:
    from trl.trainer import PPOConfig, PPOTrainer

from src.execution.executor import run_code
from src.rewards.execution_reward import compute_reward

def run_ppo_training(
    sft_model_path: str,
    tokenizer,
    dataset,
    output_dir: str = "./checkpoints/ppo",
    num_epochs: int = 1,
    learning_rate: float = 1e-6,
    batch_size: int = 4,
    mini_batch_size: int = 1,
    gradient_accumulation_steps: int = 4,
    init_kl_coef: float = 0.02,
    target_kl: float = 6.0,
    max_steps: int = 10,
):
    print("-> [1/4] Preparing PPO dataset and tokenizer...", flush=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    def tokenize_ppo_prompt(example):
        problem = example.get("question", example.get("prompt", ""))
        prompt_text = f"### Problem:\n{problem}\n\n### Solution:\n```python\n"
        tokens = tokenizer(prompt_text, truncation=True, max_length=512, padding="max_length")
        return {"input_ids": tokens["input_ids"]}

    if "input_ids" not in dataset.column_names:
        dataset = dataset.map(tokenize_ppo_prompt, remove_columns=dataset.column_names)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"-> [2/4] Loading model '{sft_model_path}' with Value Head into VRAM...", flush=True)
    ppo_model = AutoModelForCausalLMWithValueHead.from_pretrained(
        sft_model_path,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        trust_remote_code=True,
    )

    if hasattr(ppo_model, "config"):
        ppo_model.config.pad_token_id = tokenizer.pad_token_id
    if hasattr(ppo_model, "generation_config") and ppo_model.generation_config is not None:
        ppo_model.generation_config.pad_token_id = tokenizer.pad_token_id

    print("-> [3/4] Initializing TRL PPOTrainer...", flush=True)
    ppo_config = PPOConfig(
        model_name=sft_model_path,
        learning_rate=learning_rate,
        batch_size=batch_size,
        mini_batch_size=mini_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        kl_penalty="kl",
        init_kl_coef=init_kl_coef,
        target_kl=target_kl,
    )

    ppo_trainer = PPOTrainer(
        config=ppo_config,
        model=ppo_model,
        ref_model=None,
        tokenizer=tokenizer,
        dataset=dataset,
    )

    generation_kwargs = {
        "max_new_tokens": 128,
        "do_sample": True,
        "top_p": 0.95,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    step_count = 0
    total_batches = min(len(ppo_trainer.dataloader), max_steps) if max_steps else len(ppo_trainer.dataloader)

    print(f"-> [4/4] Starting PPO Rollout Optimization ({total_batches} steps max)...", flush=True)
    for epoch in range(num_epochs):
        for batch in ppo_trainer.dataloader:
            step_count += 1
            print(f"   [Step {step_count}/{total_batches}] Generating code & executing in sandbox...", flush=True)

            query_tensors = [q for q in batch["input_ids"]]
            response_tensors = ppo_trainer.generate(
                query_tensors,
                **generation_kwargs,
            )

            rewards = []
            for q, r in zip(query_tensors, response_tensors):
                code = tokenizer.decode(r, skip_special_tokens=True)
                result = run_code(code)
                reward_val = compute_reward(result["status"], 0, 1)
                rewards.append(torch.tensor(reward_val, dtype=torch.float32))

            stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
            mean_score = stats.get("ppo/mean_scores", 0.0)
            kl_val = stats.get("objective/kl", 0.0)
            print(f"   ✓ Completed Step {step_count}/{total_batches} | mean_reward={mean_score:.3f} | kl={kl_val:.3f}", flush=True)

            if max_steps and step_count >= max_steps:
                break
        if max_steps and step_count >= max_steps:
            break

    print(f"-> Saving final PPO adapter checkpoint to {output_dir}/final...", flush=True)
    ppo_model.save_pretrained(f"{output_dir}/final")
    tokenizer.save_pretrained(f"{output_dir}/final")
    return ppo_trainer

for var in ['model', 'base_model', 'reloaded_model', 'trainer', 'ppo_model']:
    if var in globals():
        del globals()[var]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

session_start = time.time()
print("Starting PPO Reinforcement Learning Training...", flush=True)

ppo_trainer = run_ppo_training(
    sft_model_path=effective_model,
    tokenizer=tokenizer,
    dataset=ppo_dataset,
    output_dir="./checkpoints/ppo",
    num_epochs=1,
    learning_rate=1e-6,
    batch_size=4,
    mini_batch_size=1,
    gradient_accumulation_steps=4,
    init_kl_coef=0.02,
    target_kl=6.0,
    max_steps=10,
)

print("\nPPO Training completed successfully!", flush=True)
print("Saved final PPO adapter checkpoint to ./checkpoints/ppo/final", flush=True)

## Step 4: Checkpoint Verification & Inference Test
Verifies saved PPO adapter files, reloads weights, confirms LoRA config, and executes 3 APPS inference tests.

In [ ]:
import os
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

checkpoint_dir = None
search_roots = ['./checkpoints/ppo/final', '/kaggle/working/checkpoints/ppo/final', '/kaggle/working', '/kaggle/input']

for root in search_roots:
    if os.path.exists(root):
        if os.path.exists(os.path.join(root, 'adapter_config.json')):
            checkpoint_dir = root
            break
        for r, dirs, files in os.walk(root):
            if 'adapter_config.json' in files and 'ppo' in r.lower():
                checkpoint_dir = r
                break
    if checkpoint_dir:
        break

if not checkpoint_dir:
    checkpoint_dir = "./checkpoints/ppo/final"

print(f"=== 1. Inspecting PPO Checkpoint Files in {checkpoint_dir} ===")
if os.path.exists(checkpoint_dir):
    for fname in sorted(os.listdir(checkpoint_dir)):
        fpath = os.path.join(checkpoint_dir, fname)
        if os.path.isfile(fpath):
            size_mb = os.path.getsize(fpath) / (1024 * 1024)
            print(f"  - {fname}: {size_mb:.2f} MB")

    print("\n=== 2. Verifying PPO LoRA Configuration ===")
    config = PeftConfig.from_pretrained(checkpoint_dir)
    print(f"  - lora_alpha: {config.lora_alpha}")
    print(f"  - r: {config.r}")
    print(f"  - target_modules: {list(config.target_modules)}")
    print(f"  - peft_type: {config.peft_type}")

    print("\n=== 3. Reloading Saved PPO Model & Adapter ===")
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto" if torch.cuda.is_available() else None,
        trust_remote_code=True,
    )
    reloaded_ppo = PeftModel.from_pretrained(base_model, checkpoint_dir)
    reloaded_ppo.eval()
    print("Reloaded PPO adapter model successfully!")

    print("\n=== 4. Running APPS Inference Check (3 Examples) ===")
    sample_prompts = [
        apps_clean[0]['question'],
        apps_clean[1]['question'],
        apps_clean[2]['question']
    ]

    for idx, p in enumerate(sample_prompts):
        prompt_text = f"### Problem:\n{p[:300]}\n\n### Solution:\n```python\n"
        inputs = tokenizer(prompt_text, return_tensors="pt").to(reloaded_ppo.device)
        with torch.no_grad():
            out = reloaded_ppo.generate(**inputs, max_new_tokens=100, do_sample=False)
        gen_text = tokenizer.decode(out[0], skip_special_tokens=True)
        print(f"\n--- PPO Inference Sample {idx + 1} ---")
        print(gen_text[:250] + "...")

    print("\nCheckpoint verification completed successfully: adapter files, LoRA configuration, model reload, and inference generation verified.")